In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.decomposition import PCA
from transformers import CLIPModel, CLIPProcessor

seed = 2026
np.random.seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BASE = '/kaggle/input/competitions/shuffled-aicc-round-9'

In [ ]:
model = CLIPModel.from_pretrained(f'{BASE}/clip')

Vp = model.vision_model.embeddings.position_embedding.weight.data[1:].numpy()
Tp = model.text_model.embeddings.position_embedding.weight.data.numpy()

G, M, N = 14, 196, 77
print(f'vision {Vp.shape}, text {Tp.shape}')

## Vision

Two principal components, each ranked into 14 bins and read as a row index and a column index.

In [ ]:
Z = PCA(n_components=2, random_state=seed).fit_transform(Vp)
row = np.argsort(np.argsort(Z[:, 0])) * G // M
col = np.argsort(np.argsort(Z[:, 1])) * G // M
vpred = row * G + col

## Text

First principal component, ranked into slots.

In [ ]:
z = PCA(n_components=1, random_state=seed).fit_transform(Tp)[:, 0]
tpred = np.argsort(np.argsort(z))

In [ ]:
sub = pd.DataFrame({
    'row_id': [f'vision_{i}' for i in range(M)] + [f'text_{i}' for i in range(N)],
    'position': list(vpred) + list(tpred),
})
sub.to_csv('submission.csv', index=False)
print(sub.shape)
sub.head()

## Local check (optional, not scored)

The 100 pairs under `data` are matched: image i goes with caption i. Restore the tables with your predicted order and see how many the model lines back up. A correct recovery reaches about 0.99. A wrong order lands anywhere from roughly 0.3 to 0.7 depending on which wrong order it is,

In [ ]:
proc = CLIPProcessor.from_pretrained(f'{BASE}/clip')
pairs = pd.read_csv(f'{BASE}/data/pairs.csv')
imgs = [Image.open(f'{BASE}/data/images/{i}.jpg').convert('RGB') for i in pairs.pair_id]
pix = proc(images=imgs, return_tensors='pt')['pixel_values'].to(device)
tok = proc(text=list(pairs.caption), return_tensors='pt', padding=True, truncation=True, max_length=77).to(device)

V = model.vision_model.embeddings.position_embedding.weight.data
T = model.text_model.embeddings.position_embedding.weight.data
V[1:] = V[1:][torch.argsort(torch.as_tensor(vpred))]        # place shuffled row i at its predicted index
T[:] = T[torch.argsort(torch.as_tensor(tpred))]
model.to(device)

with torch.inference_mode():
    imf = model.visual_projection(model.vision_model(pixel_values=pix).pooler_output)
    txf = model.text_projection(model.text_model(**tok).pooler_output)
    imf = imf / imf.norm(dim=-1, keepdim=True)
    txf = txf / txf.norm(dim=-1, keepdim=True)
    i2t = (imf @ txf.T).argmax(1).cpu()

print(f'I2T R@1 after restoring with your prediction: {(i2t == torch.arange(len(pairs))).float().mean():.3f}')